# 04 - Model Architecture

All ConvNeXt-Large model definitions: Binary (2-class), Multiclass (6-class), and Ablation variants (5×5, 3×3, 9×9, Even Kernels).

In [4]:
import torch
import torch.nn as nn
from torchvision.models import convnext_large, ConvNeXt_Large_Weights


class ConvNeXtLargeCustom(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        # Load pretrained model
        self.backbone = convnext_large(weights=ConvNeXt_Large_Weights.IMAGENET1K_V1)

        # Remove original classifier
        self.backbone.classifier = nn.Identity()

        # Add pooling manually (VERY IMPORTANT!)
        self.global_pool = nn.AdaptiveAvgPool2d(1)

        feature_dim = 1536  # ConvNeXt-Large always outputs 1536

        self.classifier = nn.Sequential(
            nn.Linear(feature_dim, 512),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(512, 512),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.backbone.features(x)  # (B,1536,H,W)

        x = self.global_pool(x)        # (B,1536,1,1)
        x = x.flatten(1)               # (B,1536)

        logits = self.classifier(x)
        return logits


In [2]:
import torch
import torch.nn as nn
from torchvision.models import convnext_large, ConvNeXt_Large_Weights


class ConvNeXtLargeCustom(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()

        # Load pretrained model
        self.backbone = convnext_large(weights=ConvNeXt_Large_Weights.IMAGENET1K_V1)

        # Remove original classifier
        self.backbone.classifier = nn.Identity()

        # Add pooling manually (VERY IMPORTANT!)
        self.global_pool = nn.AdaptiveAvgPool2d(1)

        feature_dim = 1536  # ConvNeXt-Large always outputs 1536

        self.classifier = nn.Sequential(
            nn.Linear(feature_dim, 512),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(512, 512),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.backbone.features(x)  # (B,1536,H,W)

        x = self.global_pool(x)        # (B,1536,1,1)
        x = x.flatten(1)               # (B,1536)

        logits = self.classifier(x)
        return logits


In [3]:
import torch
import torch.nn as nn
from torchvision.models import convnext_large, ConvNeXt_Large_Weights

class ConvNeXtLargeCustom(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        # 1. Load the standard pretrained model (Kernel = 7x7)
        print("Loading ConvNeXt-Large (Default 7x7)...")
        self.backbone = convnext_large(weights=ConvNeXt_Large_Weights.IMAGENET1K_V1)

        # 2. PERFORM ARCHITECTURE SURGERY (7x7 -> 5x5)
        # We must do this BEFORE defining the classifier
        self._replace_7x7_with_5x5(self.backbone)

        # 3. Remove original classifier
        self.backbone.classifier = nn.Identity()

        # 4. Add pooling manually
        self.global_pool = nn.AdaptiveAvgPool2d(1)

        # ConvNeXt-Large always outputs 1536 features
        feature_dim = 1536 

        # 5. Define your custom classifier
        self.classifier = nn.Sequential(
            nn.Linear(feature_dim, 512),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(512, 512),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def _replace_7x7_with_5x5(self, module):
        """
        Recursively finds all 7x7 Conv2d layers and replaces them with 5x5.
        Adjusts padding from 3 to 2 to maintain spatial dimensions.
        """
        replaced_count = 0
        
        # We walk through the model's named modules
        for name, child in module.named_children():
            
            # If we find a Conv2d layer
            if isinstance(child, nn.Conv2d):
                # Check if it is the specific 7x7 depthwise layer
                if child.kernel_size == (7, 7) or child.kernel_size == 7:
                    
                    # Create the NEW 5x5 layer
                    new_layer = nn.Conv2d(
                        in_channels=child.in_channels,
                        out_channels=child.out_channels,
                        kernel_size=(5, 5),          # <--- CHANGED
                        stride=child.stride,
                        padding=(2, 2),              # <--- CHANGED (Crucial: 3->2)
                        groups=child.groups,         # Keep it depthwise
                        bias=(child.bias is not None)
                    )
                    
                    # OPTIONAL: Weight Initialization Strategy
                    # Since we are changing the shape, we can't perfectly copy weights.
                    # Strategy A: Random Init (Standard for Ablation) -> We do nothing, it's already random.
                    # Strategy B: Center Crop (Advanced) -> Keeps some pretrained knowledge.
                    # We will use Strategy B (Center Crop) to make convergence faster.
                    with torch.no_grad():
                        # Crop the center 5x5 from the original 7x7 weights
                        # Weights shape: (Out, In/Groups, kH, kW)
                        new_layer.weight[:] = child.weight[:, :, 1:-1, 1:-1]
                        if child.bias is not None:
                            new_layer.bias[:] = child.bias

                    # Replace the layer in the parent module
                    setattr(module, name, new_layer)
                    replaced_count += 1
                    print(f"   -> Swapped layer '{name}' to 5x5 (Padding=2)")

            else:
                # If it's a container (like Sequential or Block), recurse into it
                self._replace_7x7_with_5x5(child)

    def forward(self, x):
        x = self.backbone.features(x)  # (B, 1536, H, W)
        x = self.global_pool(x)        # (B, 1536, 1, 1)
        x = x.flatten(1)               # (B, 1536)
        logits = self.classifier(x)
        return logits

# --- QUICK TEST TO VERIFY ---
if __name__ == "__main__":
    model2 = ConvNeXtLargeCustom(num_classes=2)
    
    # Verify the first layer of the first block is now 5x5
    # Standard path: model.backbone.features[0][0] is the stem (4x4), ignore that.
    # The first STAGE is features[1]. The first BLOCK is features[1][0].
    # Inside the block, the depthwise conv is usually the first layer or inside a sequential.
    
    print("\n--- VERIFICATION ---")
    # This path depends slightly on torchvision version, but usually:
    sample_layer = model2.backbone.features[1][0].block[0] 
    print(f"Target Kernel Size: {sample_layer.kernel_size}")
    print(f"Target Padding:     {sample_layer.padding}")
    
    if sample_layer.kernel_size == (5,5) and sample_layer.padding == (2,2):
        print("SUCCESS: Model successfully converted to 5x5.")
    else:
        print("FAIL: Still using original kernels.")

Loading ConvNeXt-Large (Default 7x7)...
   -> Swapped layer '0' to 5x5 (Padding=2)
   -> Swapped layer '0' to 5x5 (Padding=2)
   -> Swapped layer '0' to 5x5 (Padding=2)
   -> Swapped layer '0' to 5x5 (Padding=2)
   -> Swapped layer '0' to 5x5 (Padding=2)
   -> Swapped layer '0' to 5x5 (Padding=2)
   -> Swapped layer '0' to 5x5 (Padding=2)
   -> Swapped layer '0' to 5x5 (Padding=2)
   -> Swapped layer '0' to 5x5 (Padding=2)
   -> Swapped layer '0' to 5x5 (Padding=2)
   -> Swapped layer '0' to 5x5 (Padding=2)
   -> Swapped layer '0' to 5x5 (Padding=2)
   -> Swapped layer '0' to 5x5 (Padding=2)
   -> Swapped layer '0' to 5x5 (Padding=2)
   -> Swapped layer '0' to 5x5 (Padding=2)
   -> Swapped layer '0' to 5x5 (Padding=2)
   -> Swapped layer '0' to 5x5 (Padding=2)
   -> Swapped layer '0' to 5x5 (Padding=2)
   -> Swapped layer '0' to 5x5 (Padding=2)
   -> Swapped layer '0' to 5x5 (Padding=2)
   -> Swapped layer '0' to 5x5 (Padding=2)
   -> Swapped layer '0' to 5x5 (Padding=2)
   -> Swapped 

In [5]:
import torch
import torch.nn as nn
from torchvision.models import convnext_large, ConvNeXt_Large_Weights

class ConvNeXtLarge3x3(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        print("Loading ConvNeXt-Large (Converting 7x7 -> 3x3)...")
        # 1. Load Pretrained
        self.backbone = convnext_large(weights=ConvNeXt_Large_Weights.IMAGENET1K_V1)

        # 2. SURGERY: Convert to 3x3
        self._replace_7x7_with_3x3(self.backbone)

        # 3. Custom Head
        self.backbone.classifier = nn.Identity()
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        
        feature_dim = 1536 
        self.classifier = nn.Sequential(
            nn.Linear(feature_dim, 512),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(512, 512),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def _replace_7x7_with_3x3(self, module):
        for name, child in module.named_children():
            if isinstance(child, nn.Conv2d):
                # Identify the depthwise 7x7 layer
                if child.kernel_size == (7, 7) or child.kernel_size == 7:
                    
                    new_layer = nn.Conv2d(
                        in_channels=child.in_channels,
                        out_channels=child.out_channels,
                        kernel_size=(3, 3),      # <--- 3x3
                        stride=child.stride,
                        padding=(1, 1),          # <--- PADDING 1 for 3x3
                        groups=child.groups,
                        bias=(child.bias is not None)
                    )
                    
                    # WEIGHT TRANSFER: Center Crop
                    # 7x7 indices are 0,1,2,3,4,5,6. The center 3x3 are indices 2,3,4.
                    # Python slice 2:5 captures indices 2,3,4.
                    with torch.no_grad():
                        new_layer.weight[:] = child.weight[:, :, 2:5, 2:5]
                        if child.bias is not None:
                            new_layer.bias[:] = child.bias

                    setattr(module, name, new_layer)
                    print(f"   -> Swapped layer '{name}' to 3x3 (Padding=1)")
            else:
                self._replace_7x7_with_3x3(child)

    def forward(self, x):
        x = self.backbone.features(x)
        x = self.global_pool(x)
        x = x.flatten(1)
        return self.classifier(x)

In [6]:
class ConvNeXtLarge9x9(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        print("Loading ConvNeXt-Large (Converting 7x7 -> 9x9)...")
        self.backbone = convnext_large(weights=ConvNeXt_Large_Weights.IMAGENET1K_V1)

        # 2. SURGERY: Convert to 9x9
        self._replace_7x7_with_9x9(self.backbone)

        self.backbone.classifier = nn.Identity()
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        
        feature_dim = 1536 
        self.classifier = nn.Sequential(
            nn.Linear(feature_dim, 512),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(512, 512),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def _replace_7x7_with_9x9(self, module):
        for name, child in module.named_children():
            if isinstance(child, nn.Conv2d):
                if child.kernel_size == (7, 7) or child.kernel_size == 7:
                    
                    new_layer = nn.Conv2d(
                        in_channels=child.in_channels,
                        out_channels=child.out_channels,
                        kernel_size=(9, 9),      # <--- 9x9
                        stride=child.stride,
                        padding=(4, 4),          # <--- PADDING 4 for 9x9
                        groups=child.groups,
                        bias=(child.bias is not None)
                    )
                    
                    # WEIGHT TRANSFER: Center Pad
                    # We have 7x7 weights, we need 9x9.
                    # We put the 7x7 in the center of the 9x9 grid (leaving a 1-pixel border of zeros)
                    with torch.no_grad():
                        new_layer.weight.zero_() # Start with zeros
                        # 9x9 indices: 0,1,2,3,4,5,6,7,8. Center 7 is 1..7 (slice 1:8)
                        new_layer.weight[:, :, 1:8, 1:8] = child.weight
                        
                        if child.bias is not None:
                            new_layer.bias[:] = child.bias

                    setattr(module, name, new_layer)
                    print(f"   -> Swapped layer '{name}' to 9x9 (Padding=4)")
            else:
                self._replace_7x7_with_9x9(child)

    def forward(self, x):
        x = self.backbone.features(x)
        x = self.global_pool(x)
        x = x.flatten(1)
        return self.classifier(x)

In [8]:


import os
import copy
import numpy as np
from PIL import Image
from contextlib import contextmanager
import itertools
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score
)
from sklearn.calibration import calibration_curve



# ===================== DEVICE =====================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ===================== LOAD DATA =====================
val_data = np.load("val_data.npy")
val_label = np.load("val_label.npy")

if val_data.ndim == 4 and val_data.shape[-1] == 3:
    val_data = val_data.transpose(0, 3, 1, 2)

x = torch.tensor(val_data, dtype=torch.float32)
y = torch.tensor(val_label, dtype=torch.long)

if x.max() > 2.0:
    x = x / 255.0

x = F.interpolate(x, size=(256, 256), mode="bilinear", align_corners=False)

mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
x = (x - mean) / std

x = x.to(device)
y = y.to(device)

val_loader = DataLoader(
    TensorDataset(x, y),
    batch_size=64,
    shuffle=False
)

# ===================== EVAL FUNCTION =====================
def evaluate_model(model, loader, name="Model"):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []

    with torch.no_grad():
        for xb, yb in loader:
            out = model(xb)
            probs = torch.softmax(out, dim=1)[:, 1]
            preds = torch.argmax(out, dim=1)

            all_labels.extend(yb.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    all_labels = np.array(all_labels)
    all_preds  = np.array(all_preds)
    all_probs  = np.array(all_probs)

    print(f"\n================ {name} =================")
    print(f"Accuracy: {accuracy_score(all_labels, all_preds)*100:.2f}%\n")
    print("Classification Report:\n")
    print(classification_report(all_labels, all_preds))
    print("Confusion Matrix:\n")
    print(confusion_matrix(all_labels, all_preds))

    return all_labels, all_probs

# ===================== LOAD MODELS =====================
model_3x3 = ConvNeXtLarge3x3(num_classes=2)
model_3x3.load_state_dict(torch.load("best_convnext_kernel3.pth", map_location=device))
model_3x3.to(device)

model_9x9 = ConvNeXtLarge9x9(num_classes=2)
model_9x9.load_state_dict(torch.load("best_convnext_kernel9.pth", map_location=device))
model_9x9.to(device)

# ===================== RUN EVAL =====================
labels_3x3, probs_3x3 = evaluate_model(model_3x3, val_loader, "ConvNeXt-Large-3x3")
labels_9x9, probs_9x9 = evaluate_model(model_9x9, val_loader, "ConvNeXt-Large-9x9")

# ===================== ROC CURVE =====================
fpr3, tpr3, _ = roc_curve(labels_3x3, probs_3x3)
fpr9, tpr9, _ = roc_curve(labels_9x9, probs_9x9)

auc3 = auc(fpr3, tpr3)
auc9 = auc(fpr9, tpr9)

plt.figure()
plt.plot(fpr3, tpr3, label=f"3x3 AUC={auc3:.3f}")
plt.plot(fpr9, tpr9, label=f"9x9 AUC={auc9:.3f}")
plt.plot([0,1],[0,1],'--')
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("ROC Curve Comparison")
plt.legend()
plt.grid()
plt.show()

# ===================== PR CURVE =====================
p3, r3, _ = precision_recall_curve(labels_3x3, probs_3x3)
p9, r9, _ = precision_recall_curve(labels_9x9, probs_9x9)

ap3 = average_precision_score(labels_3x3, probs_3x3)
ap9 = average_precision_score(labels_9x9, probs_9x9)

plt.figure()
plt.plot(r3, p3, label=f"3x3 AP={ap3:.3f}")
plt.plot(r9, p9, label=f"9x9 AP={ap9:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Curve Comparison")
plt.legend()
plt.grid()
plt.show()

# ===================== CALIBRATION CURVE =====================
pt3, pp3 = calibration_curve(labels_3x3, probs_3x3, n_bins=10)
pt9, pp9 = calibration_curve(labels_9x9, probs_9x9, n_bins=10)

plt.figure()
plt.plot(pp3, pt3, marker="o", label="3x3")
plt.plot(pp9, pt9, marker="o", label="9x9")
plt.plot([0,1],[0,1],'--')
plt.xlabel("Predicted Probability")
plt.ylabel("True Probability")
plt.title("Calibration Curve Comparison")
plt.legend()
plt.grid()
plt.show()


Loading ConvNeXt-Large (Converting 7x7 -> 3x3)...
   -> Swapped layer '0' to 3x3 (Padding=1)
   -> Swapped layer '0' to 3x3 (Padding=1)
   -> Swapped layer '0' to 3x3 (Padding=1)
   -> Swapped layer '0' to 3x3 (Padding=1)
   -> Swapped layer '0' to 3x3 (Padding=1)
   -> Swapped layer '0' to 3x3 (Padding=1)
   -> Swapped layer '0' to 3x3 (Padding=1)
   -> Swapped layer '0' to 3x3 (Padding=1)
   -> Swapped layer '0' to 3x3 (Padding=1)
   -> Swapped layer '0' to 3x3 (Padding=1)
   -> Swapped layer '0' to 3x3 (Padding=1)
   -> Swapped layer '0' to 3x3 (Padding=1)
   -> Swapped layer '0' to 3x3 (Padding=1)
   -> Swapped layer '0' to 3x3 (Padding=1)
   -> Swapped layer '0' to 3x3 (Padding=1)
   -> Swapped layer '0' to 3x3 (Padding=1)
   -> Swapped layer '0' to 3x3 (Padding=1)
   -> Swapped layer '0' to 3x3 (Padding=1)
   -> Swapped layer '0' to 3x3 (Padding=1)
   -> Swapped layer '0' to 3x3 (Padding=1)
   -> Swapped layer '0' to 3x3 (Padding=1)
   -> Swapped layer '0' to 3x3 (Padding=1)
   -

/tmp/ipykernel_345547/238390080.py:88: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_3x3.load_state_dict(torch.load("best_convnext_kernel3.pth", map_location=device))


Loading ConvNeXt-Large (Converting 7x7 -> 9x9)...
   -> Swapped layer '0' to 9x9 (Padding=4)
   -> Swapped layer '0' to 9x9 (Padding=4)
   -> Swapped layer '0' to 9x9 (Padding=4)
   -> Swapped layer '0' to 9x9 (Padding=4)
   -> Swapped layer '0' to 9x9 (Padding=4)
   -> Swapped layer '0' to 9x9 (Padding=4)
   -> Swapped layer '0' to 9x9 (Padding=4)
   -> Swapped layer '0' to 9x9 (Padding=4)
   -> Swapped layer '0' to 9x9 (Padding=4)
   -> Swapped layer '0' to 9x9 (Padding=4)
   -> Swapped layer '0' to 9x9 (Padding=4)
   -> Swapped layer '0' to 9x9 (Padding=4)
   -> Swapped layer '0' to 9x9 (Padding=4)
   -> Swapped layer '0' to 9x9 (Padding=4)
   -> Swapped layer '0' to 9x9 (Padding=4)
   -> Swapped layer '0' to 9x9 (Padding=4)
   -> Swapped layer '0' to 9x9 (Padding=4)
   -> Swapped layer '0' to 9x9 (Padding=4)
   -> Swapped layer '0' to 9x9 (Padding=4)
   -> Swapped layer '0' to 9x9 (Padding=4)
   -> Swapped layer '0' to 9x9 (Padding=4)
   -> Swapped layer '0' to 9x9 (Padding=4)
   -

/tmp/ipykernel_345547/238390080.py:92: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_9x9.load_state_dict(torch.load("best_convnext_kernel9.pth", map_location=device))



================ ConvNeXt-Large-3x3 =================
Accuracy: 90.17%

Classification Report:

              precision    recall  f1-score   support

           0       0.96      0.84      0.90      7544
           1       0.86      0.96      0.91      7456

    accuracy                           0.90     15000
   macro avg       0.91      0.90      0.90     15000
weighted avg       0.91      0.90      0.90     15000

Confusion Matrix:

[[6348 1196]
 [ 279 7177]]

================ ConvNeXt-Large-9x9 =================
Accuracy: 87.92%

Classification Report:

              precision    recall  f1-score   support

           0       0.97      0.78      0.87      7544
           1       0.82      0.98      0.89      7456

    accuracy                           0.88     15000
   macro avg       0.89      0.88      0.88     15000
weighted avg       0.89      0.88      0.88     15000

Confusion Matrix:

[[5896 1648]
 [ 164 7292]]


NameError: name 'plt' is not defined

In [3]:
# --- QUICK TEST FOR 3x3 ---
if __name__ == "__main__":
    # Initialize the model
    model3 = ConvNeXtLarge3x3(num_classes=2)
    
    print("\n--- VERIFICATION (3x3) ---")
    # Access the first depthwise conv layer in the first stage
    # Path: features[1] -> Block[0] -> block[0] (Conv2d)
    sample_layer = model3.backbone.features[1][0].block[0]
    
    print(f"Target Kernel Size: {sample_layer.kernel_size}") # Should be (3, 3)
    print(f"Target Padding:     {sample_layer.padding}")     # Should be (1, 1)
    
    if sample_layer.kernel_size == (3,3) and sample_layer.padding == (1,1):
        print("SUCCESS: Model successfully converted to 3x3.")
    else:
        print(f"FAIL: Kernel is {sample_layer.kernel_size}, Padding is {sample_layer.padding}")

Loading ConvNeXt-Large (Converting 7x7 -> 3x3)...
   -> Swapped layer '0' to 3x3 (Padding=1)
   -> Swapped layer '0' to 3x3 (Padding=1)
   -> Swapped layer '0' to 3x3 (Padding=1)
   -> Swapped layer '0' to 3x3 (Padding=1)
   -> Swapped layer '0' to 3x3 (Padding=1)
   -> Swapped layer '0' to 3x3 (Padding=1)
   -> Swapped layer '0' to 3x3 (Padding=1)
   -> Swapped layer '0' to 3x3 (Padding=1)
   -> Swapped layer '0' to 3x3 (Padding=1)
   -> Swapped layer '0' to 3x3 (Padding=1)
   -> Swapped layer '0' to 3x3 (Padding=1)
   -> Swapped layer '0' to 3x3 (Padding=1)
   -> Swapped layer '0' to 3x3 (Padding=1)
   -> Swapped layer '0' to 3x3 (Padding=1)
   -> Swapped layer '0' to 3x3 (Padding=1)
   -> Swapped layer '0' to 3x3 (Padding=1)
   -> Swapped layer '0' to 3x3 (Padding=1)
   -> Swapped layer '0' to 3x3 (Padding=1)
   -> Swapped layer '0' to 3x3 (Padding=1)
   -> Swapped layer '0' to 3x3 (Padding=1)
   -> Swapped layer '0' to 3x3 (Padding=1)
   -> Swapped layer '0' to 3x3 (Padding=1)
   -

In [5]:
# --- QUICK TEST FOR 9x9 ---
if __name__ == "__main__":
    # Initialize the model
    model9 = ConvNeXtLarge9x9(num_classes=2)
    
    print("\n--- VERIFICATION (9x9) ---")
    # Access the first depthwise conv layer in the first stage
    sample_layer = model9.backbone.features[1][0].block[0] 
    
    print(f"Target Kernel Size: {sample_layer.kernel_size}") # Should be (9, 9)
    print(f"Target Padding:     {sample_layer.padding}")     # Should be (4, 4)
    
    if sample_layer.kernel_size == (9,9) and sample_layer.padding == (4,4):
        print("SUCCESS: Model successfully converted to 9x9.")
    else:
        print(f"FAIL: Kernel is {sample_layer.kernel_size}, Padding is {sample_layer.padding}")

Loading ConvNeXt-Large (Converting 7x7 -> 9x9)...
   -> Swapped layer '0' to 9x9 (Padding=4)
   -> Swapped layer '0' to 9x9 (Padding=4)
   -> Swapped layer '0' to 9x9 (Padding=4)
   -> Swapped layer '0' to 9x9 (Padding=4)
   -> Swapped layer '0' to 9x9 (Padding=4)
   -> Swapped layer '0' to 9x9 (Padding=4)
   -> Swapped layer '0' to 9x9 (Padding=4)
   -> Swapped layer '0' to 9x9 (Padding=4)
   -> Swapped layer '0' to 9x9 (Padding=4)
   -> Swapped layer '0' to 9x9 (Padding=4)
   -> Swapped layer '0' to 9x9 (Padding=4)
   -> Swapped layer '0' to 9x9 (Padding=4)
   -> Swapped layer '0' to 9x9 (Padding=4)
   -> Swapped layer '0' to 9x9 (Padding=4)
   -> Swapped layer '0' to 9x9 (Padding=4)
   -> Swapped layer '0' to 9x9 (Padding=4)
   -> Swapped layer '0' to 9x9 (Padding=4)
   -> Swapped layer '0' to 9x9 (Padding=4)
   -> Swapped layer '0' to 9x9 (Padding=4)
   -> Swapped layer '0' to 9x9 (Padding=4)
   -> Swapped layer '0' to 9x9 (Padding=4)
   -> Swapped layer '0' to 9x9 (Padding=4)
   -